In [3]:
#### PICK ONE: This node for temporary install
SWARMPATH = '/content/'

In [ ]:
#### PICK ONE: *OR* this node for install in google drive
from google.colab import drive
drive.mount('/content/drive')
SWARMPATH = '/content/drive/MyDrive'

# Colab breaks venvs, and doesn't save anything valid to drive, so just screw it do a global install of pip reqs if we used a comfy backend in the drive
!if [[ -f "$SWARMPATH/SwarmUI/dlbackend/ComfyUI/requirements.txt" ]]; then rm -rf $SWARMPATH/SwarmUI/dlbackend/ComfyUI/venv/; pip install -r $SWARMPATH/SwarmUI/dlbackend/ComfyUI/requirements.txt; fi

In [4]:
# Install dotnet dependencies
!wget https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

# Install Clouldflared
!wget https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# use the chosen path
import os
os.environ['SWARMPATH'] = SWARMPATH
%cd $SWARMPATH

# Colab breaks venv, so, tell swarm to not make a venv
os.environ['SWARM_NO_VENV'] = 'true'

# Download SwarmUI
url = "https://github.com/mcmonkeyprojects/SwarmUI"
!git clone $url

--2026-11-13 19:45:37--  https://dot.net/v1/dotnet-install.sh
Resolving dot.net (dot.net)... 20.76.201.171, 20.112.250.133, 20.236.44.162, ...
Connecting to dot.net (dot.net)|20.76.201.171|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh [following]
--2026-11-13 19:45:38--  https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh
Resolving builds.dotnet.microsoft.com (builds.dotnet.microsoft.com)... 23.11.232.110, 23.11.232.172, 2600:1406:2e00:b00::170b:e8ac, ...
Connecting to builds.dotnet.microsoft.com (builds.dotnet.microsoft.com)|23.11.232.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/octet-stream]
Saving to: ‘dotnet-install.sh’

dotnet-install.sh       [ <=>                ]  62.07K  --.-KB/s    in 0.006s  

2026-11-13 19:45:38 (10.4 MB/s) - ‘dotnet-install.sh’ saved [63563]

dotnet-install: Attemp

In [ ]:
# Force local path configuration to bypass Drive
import os
import subprocess
import time
import re

SWARMPATH = '/content'
os.environ['SWARMPATH'] = SWARMPATH
os.environ['SWARM_NO_VENV'] = 'true'

# Clone repository locally if it does not exist
%cd /content
if not os.path.exists('/content/SwarmUI'):
    !git clone https://github.com/mcmonkeyprojects/SwarmUI

# Check if cloudflared is installed, otherwise install it
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("Installazione di Cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    !rm cloudflared-linux-amd64.deb

# Ensure dotnet is linked
!ln -sf /root/.dotnet/dotnet /usr/bin/dotnet
!ln -sf /root/.dotnet/dotnet /usr/local/bin/dotnet

# Clean and pull latest code
%cd /content/SwarmUI
!git fetch
!git reset --hard origin/master
!git pull --autostash
!rm -rf ./src/bin/live_release

# Start SwarmUI in the background on port 7801
print("Avvio di SwarmUI in background...")
swarm_proc = subprocess.Popen(
    ["bash", "./launch-linux.sh", "--launch_mode", "none", "--port", "7801"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=dict(os.environ, PATH=f"/root/.dotnet:{os.environ['PATH']}")
)

# Start Cloudflare Tunnel in background pointing to port 7801
print("Creazione del tunnel Cloudflare...")
!nohup cloudflared tunnel --url http://localhost:7801 > /content/tunnel.log 2>&1 &

time.sleep(5)

# Extract and print the public URL
print("\n=== LINK DI ACCESSO PUBBLICO ===")
for _ in range(10):
    if os.path.exists('/content/tunnel.log'):
        with open('/content/tunnel.log', 'r') as f:
            log = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log)
            if match:
                print(f"\nClicca qui per accedere a SwarmUI: {match.group(0)}\n")
                break
    time.sleep(2)
else:
    print("Impossibile rilevare automaticamente il link. Controlla il file /content/tunnel.log")

# Monitor SwarmUI output
print("Visualizzazione output SwarmUI:")
for line in iter(swarm_proc.stdout.readline, ''):
    print(line, end='')

/content
Installazione di Cloudflared...
Selecting previously unselected package cloudflared.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2024.8.2) ...
Setting up cloudflared (2024.8.2) ...
Processing triggers for man-db (2.12.0-4build2) ...
/content/SwarmUI
HEAD is now at d347f306 scrap directml ref
Already up to date.
Avvio di SwarmUI in background...
Creazione del tunnel Cloudflare...

=== LINK DI ACCESSO PUBBLICO ===

Clicca qui per accedere a SwarmUI: https://remedies-small-passport-tile.trycloudflare.com

Visualizzazione output SwarmUI:
  Determining projects to restore...
  All projects are up-to-date for restore.
  SwarmUI -> /content/SwarmUI/src/bin/live_release/SwarmUI.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:04.89
21:11:28.126 [Init] === SwarmUI v0.9.8.4 Starting at 2026-11-13 21:11:28 ===
21:11:28.132 [Init] Parsing command line...
21:11:2